# PerFedAvg Client

> "Personalized Federated Learning with Theoretical Guarantees: A Model-Agnostic Meta-Learning Approach". AS implemented in https://proceedings.neurips.cc/paper/2020/file/24389bfe4fe2eba8bf9aa9203a44cdad-Paper.pdf

In [ ]:
#| default_exp clients.perfedavg

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export

import math
from copy import deepcopy

from fastcore.utils import *
from fastcore.all import *

import torch
import torch.nn.functional as F

from tqdm import tqdm
from loguru import logger

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

## Per-FedAvg Client

In [ ]:
#| export
@AlgorithmRegistry.register_client("perfedavg")
class ClientPerFedAvg(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ): 
        
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        
        self.iter_trainloader = iter(self.train_loader)
        self.iter_testloader = iter(self.test_loader)

In [ ]:
#| export
@patch
def get_next_batch(self: ClientPerFedAvg, train= True) -> dict:
    
    loader_type = self.train_loader if train else self.test_loader
    to_iter = self.iter_trainloader if train else self.iter_testloader

    if(int(self.cfg.data.batch_size) == 0):
        for batch in loader_type:
            X = batch[self.data_key]
            y = batch[self.label_key]
            return (X.to(self.device), y.to(self.device))
        
    else:
        try:

            batch = next(to_iter)
            X = batch[self.data_key]
            y = batch[self.label_key]
            return (X.to(self.device), y.to(self.device))
        
        except StopIteration:

            if train:
                self.iter_trainloader = iter(self.train_loader)
            else:
                self.iter_testloader = iter(self.test_loader)

            # batch = next(to_iter)
            X = None
            y = None
            return (X, y)      


### Training

In [ ]:
#| export
@patch
def train_step(self: ClientPerFedAvg, num_steps= None):

    self.model = self.model.to(self.device)

    self.model.train()

    for _ in range(self.cfg.local_epochs):
        while True:
            batch = self.get_next_batch()
            X, y = batch
            if X is None or y is None:
                break

            temp_model = deepcopy(list(self.model.parameters()))
            self.optimizer.zero_grad()
            output = self.model(X)
            loss = self.criterion(output, y)
            loss.backward()
            self.optimizer.step()

            if num_steps == 1:
                for old_param, new_param in zip(self.model.parameters(), temp_model):
                    old_param.data = new_param.data.clone()
                break
            
            batch = self.get_next_batch()
            X, y = batch
            self.optimizer.zero_grad()
            output = self.model(X)
            loss = self.criterion(output, y)
            loss.backward()
            
            for old_param, new_param in zip(self.model.parameters(), temp_model):
                old_param.data = new_param.data.clone()
            

            self.optimizer.step(beta= self.cfg.algorithm.beta)


In [ ]:
#| export
@patch
def fit(self: ClientPerFedAvg) -> None:
    self.train_step()
    self.train_step()

### Evaluation

In [ ]:
#| export
@patch
def train_test_stats(self: ClientPerFedAvg, batch: dict, model) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    X, y = batch
    logits = model(X)
    loss = self.criterion(logits, y)
    y_pred = logits.argmax(dim=-1)
    y_true = y

    metrics = self.training_metrics.compute(y_pred= y_pred, y_true= y_true)

    return loss, metrics


In [ ]:
#| export
@patch
def evaluate_local(self: ClientPerFedAvg, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []

    self.model = self.model.to(self.device)
    self.train_step(num_steps= 1)
    self.model.eval()
    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.get_next_batch()
            if batch[0] is None or batch[1] is None:
                continue
            model = self.model
            loss, metrics = self.train_test_stats(batch, model)                 
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[-1])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}
        
    return {"loss": avg_loss, "metrics": total_metrics}


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()